# Chapter 4: Reflection

EliteAI Academy

Generate, critique, and revise outputs using explicit criteria, structured feedback, and stopping rules.

## 0. Setup

Run `setup.ps1` from the course folder once, then select **Python (Agentic AI Course)** as the kernel.
The course loads your DeepSeek credentials from `.env`; restart the kernel after editing that file.
The install cell below is optional if setup is already complete. Run the remaining cells in order.

**Teaching check:** explain the pattern, run one example, inspect its output, then try an exercise.
Live model calls use API credits and may produce different answers. Stock prices, weather, CRM records,
email delivery, and approval workflows in these lessons are classroom simulations. Model-generated
research is illustrative and needs independent source verification before factual use.


In [ ]:
%pip install -r requirements-lock.txt --quiet

In [ ]:
from course_config import MODEL
import os, json, re, time
from typing import Optional

PROVIDER = "deepseek"

if PROVIDER == "deepseek":
    import anthropic
    _client = anthropic.Anthropic(timeout=60.0, max_retries=2)
    _model  = MODEL

    def call_llm(prompt: str, system: str = "You are a helpful AI assistant.",
                 history: list = None) -> str:
        'Call the LLM with optional conversation history.'
        messages = (history or []) + [{"role": "user", "content": prompt}]
        r = _client.messages.create(thinking={"type": "disabled"}, 
            model=_model, max_tokens=1024, system=system, messages=messages
        )
        return r.content[0].text.strip()

elif PROVIDER == "gemini":
    import google.generativeai as genai
    genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
    _gmodel = genai.GenerativeModel("gemini-2.0-flash")

    def call_llm(prompt: str, system: str = "You are a helpful AI assistant.",
                 history: list = None) -> str:
        context = ""
        if history:
            for msg in history:
                context += f"{msg['role'].upper()}: {msg['content']}\n\n"
        full = f"{system}\n\n{context}USER: {prompt}"
        return _gmodel.generate_content(full).text.strip()

print(f"✅ Provider: {PROVIDER}")
print(call_llm("Reply with exactly: REFLECTION READY"))

## 1. The Problem — Single-Pass Outputs Are Often Suboptimal

Ask for something complex in one shot — the result is usually good enough, but not great. Reflection gives the agent a second (and third) look.

In [ ]:
TASK = """
Write a Python function called `calculate_factorial(n)` that:
1. Accepts a single integer n
2. Returns n! (factorial)
3. Handles edge case: factorial of 0 is 1
4. Raises ValueError for negative inputs
5. Includes a clear docstring
6. Has type hints
"""

single_pass = call_llm(
    TASK,
    system="You are a Python developer. Write clean, working code."
)

print("=== SINGLE-PASS OUTPUT ===")
print(single_pass)
print("\n⚠️  May be missing type hints, edge cases, or best practices.")
print("   Reflection will progressively improve this.")

## 2. The Reflection Loop Architecture

Draft → Critique → Revise → Repeat or stop. Choose the critic, evaluation criteria, and stopping limit.

In [ ]:
STOP_SIGNAL = "OUTPUT_IS_PERFECT"

def producer(task: str, previous_output: str = "",
             critique: str = "", history: list = None) -> str:
    'Generate or refine output.'
    if not previous_output:
        prompt = f"Complete this task:\n\n{task}"
    else:
        prompt = f"""Refine your previous output using the critique below.
        
Original task:
{task}

Your previous output:
{previous_output}

Critique to address:
{critique}

Produce an improved version that fully addresses all critique points."""
    return call_llm(prompt, system="You are an expert producer. Generate high-quality outputs.",
                   history=history)

def critic(task: str, output: str, criteria: str = "") -> str:
    'Evaluate output against the task requirements.'
    default_criteria = "accuracy, completeness, clarity, and adherence to requirements"
    eval_criteria = criteria or default_criteria

    prompt = f"""You are a meticulous critic. Evaluate the output below.

Original Task:
{task}

Output to Evaluate:
{output}

Evaluation Criteria: {eval_criteria}

Instructions:
- If the output fully meets all requirements with no issues, respond ONLY with: {STOP_SIGNAL}
- Otherwise, provide a concise bulleted list of specific issues and improvements needed.
- Be precise. Reference exact problems, not vague suggestions."""

    return call_llm(prompt, system="You are a senior expert reviewer. Be rigorous and precise.")

def reflection_loop(task: str, max_iterations: int = 3,
                    criteria: str = "", verbose: bool = True) -> dict:
    'Run the full reflection loop.'
    current_output = ""
    critique       = ""
    history        = []
    all_versions   = []

    for i in range(max_iterations):
        if verbose:
            print(f"\n{'='*55}")
            print(f"  ITERATION {i+1}/{max_iterations}")
            print(f"{'='*55}")

        if verbose: print(f"\n[STAGE 1] {'Generating' if i==0 else 'Refining'} output...")
        current_output = producer(task, current_output, critique, history)
        all_versions.append({"version": i+1, "output": current_output})

        if verbose:
            preview = current_output[:300] + ("..." if len(current_output) > 300 else "")
            print(f"\n--- Output v{i+1} ---\n{preview}")

        history.append({"role": "assistant", "content": current_output})

        if verbose: print(f"\n[STAGE 2] Critiquing output v{i+1}...")
        critique = critic(task, current_output, criteria)

        if STOP_SIGNAL in critique:
            if verbose: print(f"\n✅ STOP SIGNAL received — output is satisfactory.")
            break

        if verbose: print(f"\n--- Critique ---\n{critique}")

        history.append({"role": "user",
                       "content": f"Critic feedback for v{i+1}:\n{critique}"})

    return {
        "final_output": current_output,
        "iterations":   i + 1,
        "versions":     all_versions,
        "history":      history,
    }

print("✅ Reflection utilities defined: producer(), critic(), reflection_loop()")

## 3. Use Case 1 — Code Generation with Reflection

The book's core example: write a factorial function, critique it, refine it.

In [ ]:
CODE_TASK = """
Write a Python function called `calculate_factorial(n: int) -> int` that:
1. Accepts a single integer n with a type hint
2. Returns n! (factorial)
3. Handles edge case: factorial(0) = 1
4. Raises ValueError with a clear message for negative inputs
5. Includes a Google-style docstring with Args, Returns, and Raises sections
6. Is PEP 8 compliant
7. Does NOT use math.factorial (implement it manually)
"""

CODE_CRITERIA = """
- Correct type hints on all parameters and return value
- Google-style docstring with Args, Returns, Raises
- Correct handling of n=0 (must return 1)
- ValueError raised for negative input with a descriptive message
- No use of math.factorial
- PEP 8 compliance (naming, spacing)
- Clean implementation logic
"""

code_result = reflection_loop(
    task=CODE_TASK,
    max_iterations=3,
    criteria=CODE_CRITERIA,
    verbose=True
)

print(f"\n{'='*55}")
print(f"FINAL CODE (after {code_result['iterations']} iteration(s))")
print(f"{'='*55}")
print(code_result["final_output"])

## 4. Self-Reflection vs Producer-Critic — Side by Side

**Self-reflection:** one agent reviews its own output (cognitive bias risk)  
**Producer-Critic:** two specialised agents with distinct system prompts (more objective)

In [ ]:
BLOG_TASK = "Write a 150-word blog intro about why Python is great for data science."

def self_reflection(task: str, iterations: int = 2) -> str:
    output = call_llm(task, system="You are a content writer.")
    print(f"[SELF-REFLECTION] Initial output:\n{output[:200]}...\n")

    for i in range(iterations):
        review_prompt = f"""
        Review your own output below for the task: '{task}'
        Identify any issues with flow, tone, accuracy, or completeness.
        Then produce an improved version.
        
        Your previous output:
        {output}
        """
        output = call_llm(review_prompt, system="You are a self-critical content writer.")
        print(f"[SELF-REFLECTION] Iteration {i+1} output:\n{output[:200]}...\n")
    return output

def producer_critic(task: str, iterations: int = 2) -> str:

    output = call_llm(task, system="You are a creative content writer. Focus on quality writing.")
    print(f"[PRODUCER-CRITIC] Producer v1:\n{output[:200]}...\n")

    for i in range(iterations):

        critique = call_llm(
            f"Task: {task}\n\nContent to review:\n{output}\n\nProvide 3 specific improvements.",
            system="You are a senior editor at a tech publication. Be exacting and professional."
        )
        print(f"[PRODUCER-CRITIC] Critic feedback {i+1}:\n{critique[:200]}...\n")

        output = call_llm(
            f"Task: {task}\n\nYour draft:\n{output}\n\nEditor feedback:\n{critique}\n\nRevise accordingly.",
            system="You are a creative content writer. Incorporate editor feedback precisely."
        )
        print(f"[PRODUCER-CRITIC] Producer v{i+2}:\n{output[:200]}...\n")
    return output

print("=" * 55 + "\n SELF-REFLECTION APPROACH\n" + "=" * 55)
self_result = self_reflection(BLOG_TASK, iterations=2)

print("\n" + "=" * 55 + "\n PRODUCER-CRITIC APPROACH\n" + "=" * 55)
pc_result = producer_critic(BLOG_TASK, iterations=2)

print("\n=== FINAL COMPARISON ===")
print(f"\n[Self-Reflection Final]:\n{self_result}")
print(f"\n[Producer-Critic Final]:\n{pc_result}")

## 5. Use Case 2 — Content Writing with Structured Critique

A marketing email goes through structured reflection with a professional editor critic.

In [ ]:
EMAIL_TASK = """
Write a marketing email for a new AI-powered project management tool called 'FlowAI'.
Requirements:
- Subject line that creates urgency
- Opening hook (1 sentence)
- 3 key benefits in bullet points
- A clear call-to-action
- Professional but friendly tone
- Under 200 words total
"""

EMAIL_CRITERIA = """
- Subject line: creates genuine urgency, not clickbait
- Opening hook: compelling, addresses a pain point
- Benefits: specific and quantifiable where possible, not vague
- CTA: action-oriented verb, clear next step
- Tone: professional but human, no corporate jargon
- Word count: strictly under 200 words
- No exclamation mark overuse (max 2)
"""

email_result = reflection_loop(
    task=EMAIL_TASK,
    max_iterations=3,
    criteria=EMAIL_CRITERIA,
    verbose=True
)

print(f"\n{'='*55}")
print(f"FINAL EMAIL (after {email_result['iterations']} iteration(s))")
print(f"{'='*55}")
print(email_result["final_output"])

## 6. Use Case 3 — Summarisation Accuracy Reflection

Generate a summary, then critique it for factual completeness against the source document.

In [ ]:
SOURCE_DOCUMENT = """
The global electric vehicle (EV) market reached 14 million units sold in 2023, 
representing a 35% year-on-year increase. China accounted for 60% of global EV sales, 
followed by Europe at 25% and the United States at 10%. The average battery range 
improved to 420 km per charge, up from 350 km in 2021. However, charging infrastructure 
remains a challenge: only 2.7 million public charging stations exist globally against 
an estimated need of 10 million by 2030. Battery costs fell to $139/kWh, a 90% reduction 
from 2010 levels. Three manufacturers — BYD, Tesla, and Volkswagen — captured 45% of 
total market share. Government incentives averaged $3,400 per vehicle across OECD nations.
"""

SUMMARY_TASK = f"""
Summarise the following document in exactly 4 sentences.
Include the most important statistics.

Document:
{SOURCE_DOCUMENT}
"""

SUMMARY_CRITERIA = f"""
Check the summary against the original document:
- All statistics mentioned must be accurate (check numbers exactly)
- Key facts covered: market size, top regions, battery range, charging gap, battery cost, top manufacturers
- Exactly 4 sentences (count them)
- No invented statistics or hallucinations
- Concise — no redundancy

Original document for reference:
{SOURCE_DOCUMENT}
"""

summary_result = reflection_loop(
    task=SUMMARY_TASK,
    max_iterations=3,
    criteria=SUMMARY_CRITERIA,
    verbose=True
)

print(f"\n{'='*55}")
print(f"FINAL SUMMARY (after {summary_result['iterations']} iteration(s))")
print(f"{'='*55}")
print(summary_result["final_output"])

## 7. Use Case 4 — Plan Generation & Feasibility Review

An agent proposes a plan, a critic evaluates feasibility and risks, the agent revises.

In [ ]:
PLAN_TASK = """
Create a 4-week project plan to migrate a company's on-premise SQL Server database 
to Azure Synapse Analytics. The team has 2 data engineers and 1 solution architect.
Format: Week-by-week breakdown with tasks, owners, and dependencies.
"""

PLAN_CRITERIA = """
Evaluate the plan as an experienced Azure architect:
- Is 4 weeks realistic for a full migration with a 3-person team?
- Are critical phases included: assessment, schema conversion, data migration, testing, cutover?
- Are dependencies between tasks explicitly stated?
- Is risk mitigation addressed (rollback plan, data validation)?
- Are owners assigned to each task?
- Is the cutover/go-live phase in the final week?
"""

plan_result = reflection_loop(
    task=PLAN_TASK,
    max_iterations=3,
    criteria=PLAN_CRITERIA,
    verbose=True
)

print(f"\n{'='*55}")
print(f"FINAL PLAN (after {plan_result['iterations']} iteration(s))")
print(f"{'='*55}")
print(plan_result["final_output"])

## 8. LangChain LCEL Implementation — Single Reflect Cycle

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import SystemMessage, HumanMessage

lc_llm = ChatAnthropic(model_kwargs={"thinking": {"type": "disabled"}}, model=MODEL, temperature=0.1)

producer_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert Python developer. Write clean, production-ready code."),
    ("user",   "{task}")
])
producer_chain = producer_prompt | lc_llm | StrOutputParser()

critic_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a senior software engineer performing a code review.
Evaluate the code against the original task requirements.
If the code is perfect, respond ONLY with: CODE_IS_PERFECT
Otherwise, provide a bulleted list of specific issues."""),
    ("user",   "Original Task:\n{task}\n\nCode to Review:\n{code}")
])
critic_chain = critic_prompt | lc_llm | StrOutputParser()

refiner_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert Python developer. Apply the critique and improve the code."),
    ("user",   "Task:\n{task}\n\nPrevious Code:\n{code}\n\nCritique to address:\n{critique}")
])
refiner_chain = refiner_prompt | lc_llm | StrOutputParser()

def lc_reflection_loop(task: str, max_iter: int = 3) -> dict:
    print(f"🔁 Starting LangChain reflection loop (max {max_iter} iterations)")

    code     = producer_chain.invoke({"task": task})
    critique = ""

    for i in range(max_iter):
        print(f"\n[Iter {i+1}] Critiquing...")
        critique = critic_chain.invoke({"task": task, "code": code})

        if "CODE_IS_PERFECT" in critique:
            print(f"✅ CODE_IS_PERFECT — stopping at iteration {i+1}")
            break

        print(f"[Iter {i+1}] Critique:\n{critique[:300]}...")
        print(f"\n[Iter {i+1}] Refining...")
        code = refiner_chain.invoke({"task": task, "code": code, "critique": critique})

    return {"final_code": code, "last_critique": critique, "iterations": i + 1}

LC_TASK = """
Write a Python function `is_palindrome(s: str) -> bool` that:
- Checks if a string is a palindrome
- Ignores case and non-alphanumeric characters
- Has type hints, a docstring, and handles empty strings (returns True)
"""

lc_result = lc_reflection_loop(LC_TASK, max_iter=3)
print(f"\n{'='*55}")
print(f"FINAL CODE (LangChain, {lc_result['iterations']} iteration(s))")
print(f"{'='*55}")
print(lc_result["final_code"])

## 9. Tracking Quality Across Iterations

Measure how output quality improves with each reflection cycle using a scoring critic.

In [ ]:
def quality_score(task: str, output: str) -> float:
    'Ask the LLM to score the output from 1-10.'
    score_raw = call_llm(
        f"""Score the following output for the task on a scale of 1-10.
Return ONLY a single number (e.g. 7.5). Nothing else.

Task: {task}
Output: {output}""",
        system="You are a strict quality evaluator."
    )
    try:
        return float(re.search(r'\d+\.?\d*', score_raw).group())
    except:
        return 0.0

def reflection_with_scoring(task: str, criteria: str = "",
                             max_iterations: int = 3) -> None:
    'Run reflection loop and track quality score at each iteration.'
    current_output = ""
    critique       = ""
    scores         = []

    for i in range(max_iterations):
        current_output = producer(task, current_output, critique)
        score          = quality_score(task, current_output)
        scores.append(score)
        print(f"  Iteration {i+1}: Quality Score = {score:.1f}/10")

        critique = critic(task, current_output, criteria)
        if STOP_SIGNAL in critique:
            print(f"  ✅ Stopped early — output judged perfect")
            break

    print("\nQuality progression:")
    for i, s in enumerate(scores):
        bar = "█" * int(s)
        print(f"  v{i+1}: [{bar:<10}] {s:.1f}")
    print(f"\n  Improvement: {scores[0]:.1f} → {scores[-1]:.1f} "
          f"(+{scores[-1]-scores[0]:.1f} points)")

SCORE_TASK = """
Write a compelling LinkedIn post announcing a new AI product launch.
Requirements: hook in line 1, value proposition, social proof, clear CTA, 
relevant hashtags, under 300 words.
"""
print("=== QUALITY TRACKING ACROSS REFLECTION ITERATIONS ===")
reflection_with_scoring(SCORE_TASK, max_iterations=3)

## 10. Use Case 5 — Conversational Agent Self-Review

After each response in a multi-turn conversation, the agent reviews its own reply for coherence, accuracy, and alignment with conversation history.

In [ ]:
def reflective_chatbot(user_message: str, conversation_history: list) -> tuple:
    'A chatbot that reflects on its own response before delivering it.'

    history_text = "\n".join(
        f"{m['role'].upper()}: {m['content']}" for m in conversation_history
    ) if conversation_history else "(No prior conversation)"

    initial = call_llm(
        f"Conversation so far:\n{history_text}\n\nUser: {user_message}",
        system="You are a helpful customer support agent for a SaaS company."
    )

    review = call_llm(
        f"""Review your response below for quality.
        
Conversation history:
{history_text}

User's latest message: {user_message}

Your response:
{initial}

Check for:
- Does it directly address the user's question?
- Is it consistent with the conversation history?
- Is it clear and professional?

If perfect, reply: RESPONSE_IS_GOOD
Otherwise, list issues.""",
        system="You are a quality reviewer for customer support responses."
    )

    if "RESPONSE_IS_GOOD" in review:
        final_response = initial
        refined = False
    else:

        final_response = call_llm(
            f"""Improve your response addressing these issues:
{review}

Original response: {initial}
User message: {user_message}
Conversation: {history_text}""",
            system="You are a helpful customer support agent."
        )
        refined = True

    conversation_history.append({"role": "user",      "content": user_message})
    conversation_history.append({"role": "assistant", "content": final_response})

    return final_response, refined, conversation_history

history = []
turns = [
    "Hi, I'm having trouble logging into my account.",
    "I tried resetting my password but the email isn't arriving.",
    "My email is definitely correct. This is urgent — I have a demo in 30 minutes.",
]

print("=== REFLECTIVE CONVERSATIONAL AGENT ===")
for turn in turns:
    response, was_refined, history = reflective_chatbot(turn, history)
    print(f"\n👤 User: {turn}")
    print(f"🤖 Agent: {response}")
    print(f"   [Reflection applied: {'YES' if was_refined else 'NO — initial response was good'}]")
    print("-" * 60)

## 11. Cost & Latency Trade-Offs

Reflection multiplies API calls. Know when it's worth it and when it's not.

In [ ]:
trade_offs = """
╔═══════════════════════════════════╦════════════════════════════════════════════╗
║ USE Reflection When               ║ SKIP Reflection When                       ║
╠═══════════════════════════════════╬════════════════════════════════════════════╣
║ Output quality > speed            ║ Real-time responses required               ║
║ Long-form content (blogs, code,   ║ High-volume, low-stakes tasks              ║
║ reports, plans)                   ║ (e.g. classifying 10,000 tweets)           ║
║ High-stakes outputs (contracts,   ║ Budget is tight — each iteration           ║
║ medical summaries, legal docs)    ║ costs another LLM call                     ║
║ Code that will be executed        ║ Simple factual lookups                     ║
║ Nuanced creative work             ║ Routing / classification tasks             ║
╠═══════════════════════════════════╬════════════════════════════════════════════╣
║ COST FORMULA                                                                   ║
║ Total calls = N_iterations × 2 (generate + critique) + 1 (final output)       ║
║ 3 iterations = 7 LLM calls vs 1 single-pass call                              ║
╠═══════════════════════════════════╬════════════════════════════════════════════╣
║ CONTEXT WINDOW WARNING                                                         ║
║ History grows with each iteration. Watch for context window limits.            ║
║ For long tasks: summarise history before passing to next iteration.            ║
╚════════════════════════════════════════════════════════════════════════════════╝
"""
print(trade_offs)

## 12. Structured Critique Output (JSON Critique)

For programmatic control, make the critic return structured JSON instead of free text.

In [ ]:
def structured_critic(task: str, output: str) -> dict:
    'Critic that returns structured JSON critique for programmatic use.'
    prompt = f"""
    Evaluate the output below for the task provided.
    Return ONLY a JSON object with these exact keys:
    {{
      "score": <integer 1-10>,
      "is_perfect": <true or false>,
      "issues": [<list of specific issue strings>],
      "strengths": [<list of specific strength strings>],
      "priority_fix": <single string: the most important thing to fix, or null if perfect>
    }}
    
    Task: {task}
    Output to evaluate:
    {output}
    """
    raw = call_llm(prompt, system="You are a precise evaluator. Return only valid JSON.")
    clean = re.sub(r"```[\w]*", "", raw).strip()
    return json.loads(clean)

def structured_reflection_loop(task: str, max_iterations: int = 3) -> dict:
    output  = producer(task)
    history = []

    for i in range(max_iterations):
        critique_data = structured_critic(task, output)
        print(f"\n[Iteration {i+1}] Score: {critique_data['score']}/10 | "
              f"Perfect: {critique_data['is_perfect']}")
        print(f"  Issues   : {critique_data['issues']}")
        print(f"  Strengths: {critique_data['strengths']}")
        print(f"  Priority : {critique_data['priority_fix']}")

        if critique_data["is_perfect"] or critique_data["score"] >= 9:
            print("  ✅ Score threshold reached — stopping.")
            break

        output = producer(task, output,
                         f"Priority fix: {critique_data['priority_fix']}\n"
                         f"All issues: {critique_data['issues']}")

    return {"final_output": output, "final_score": critique_data["score"]}

STRUCTURED_TASK = """
Write a 3-sentence executive summary of AI's impact on healthcare in 2024,
including at least 2 specific statistics and a forward-looking statement.
"""

print("=== STRUCTURED JSON CRITIQUE REFLECTION ===")
result = structured_reflection_loop(STRUCTURED_TASK, max_iterations=3)
print(f"\n{'='*55}")
print(f"FINAL OUTPUT (Score: {result['final_score']}/10)")
print(f"{'='*55}")
print(result["final_output"])

## 13. Summary — Pattern Rules of Thumb

In [ ]:
summary = """
╔══════════════════════════════════╦═════════════════════════════════════════════╗
║ Reflection Architecture Choices  ║ Design Rules                                ║
╠══════════════════════════════════╬═════════════════════════════════════════════╣
║ Self-reflection                  ║ Good for general improvement                ║
║ (1 agent, 2 prompts)             ║ Cognitive bias risk — agent favours          ║
║                                  ║ its own style even with issues              ║
╠══════════════════════════════════╬═════════════════════════════════════════════╣
║ Producer-Critic                  ║ Better objectivity and specialisation       ║
║ (2 agents, distinct personas)    ║ Critic should have domain expertise persona ║
╠══════════════════════════════════╬═════════════════════════════════════════════╣
║ Stopping conditions              ║ Always define at least ONE stop condition:  ║
║                                  ║ signal word | max iterations | score ≥ N    ║
╠══════════════════════════════════╬═════════════════════════════════════════════╣
║ Structured critique (JSON)       ║ Use for programmatic control, dashboards,   ║
║                                  ║ or score-gated stopping                     ║
╠══════════════════════════════════╬═════════════════════════════════════════════╣
║ Conversation history             ║ Pass full history to each iteration for     ║
║                                  ║ cumulative improvement (not just last step) ║
╚══════════════════════════════════╩═════════════════════════════════════════════╝
"""
print(summary)

## 14. Exercises

- **Exercise 1 — Domain-Specific Critic:** Build a reflection loop for SQL query generation. The critic should check for: correct JOIN syntax, proper WHERE clauses, SQL injection risks, and performance (no `SELECT *`).

- **Exercise 2 — Score-Gated Stopping:** Modify `structured_reflection_loop` so it stops only when `score >= 8.5` (not on `is_perfect`). How many iterations does it take on average?

- **Exercise 3 — Multi-Criteria Critic:** Build a critic that evaluates content on 3 dimensions separately: accuracy (1-10), clarity (1-10), completeness (1-10). Stop when all three exceed 8.

- **Exercise 4 — Parallel Reflection:** Combine Chapter 3 (parallelization) + Chapter 4 (reflection): Generate 3 versions of a blog post in parallel, then run one critique pass on each, then select the highest-scoring version.

- **Exercise 5 — Reflection Memory:** Implement a simple memory store that tracks all critiques across a session. Before each new critique, inject the last 3 critiques as context so the critic avoids repeating itself.

**Next:** Chapter 5 — Tool Use & Function Calling